In [ ]:
import pymc as pm
import arviz as az
import numpy as np
from src.spline_utils import get_bspline_matrix

# 1. Prepare Data
# Let's say these are Malaysian Corporate 'AAA' Spreads
market_tenors = np.array([1, 2, 3, 5, 7, 10])
observed_spreads = np.array([85, 102, 120, 155, 180, 210]) 

# Generate the Basis Matrix X using the knots we defined in Part 1
knots = np.array([1, 3, 5, 7, 10])
X_basis = get_bspline_matrix(market_tenors, knots)

# 2. Build the Bayesian Model
with pm.Model() as credit_model:
    # Prior for smoothness: High 'tau' means more wiggle, Low 'tau' means very smooth
    tau = pm.Exponential("tau", 1.0)
    
    # The Random Walk Prior (The 'WOW' factor for AdComs)
    # This says: Each spline segment's height depends on the previous one
    beta = pm.GaussianRandomWalk("beta", sigma=tau, shape=X_basis.shape[1])
    
    # Link Function: The expected spread (mu) = X * beta
    mu = pm.math.dot(X_basis, beta)
    
    # Likelihood: Market observations follow a Normal distribution around 'mu'
    sigma = pm.HalfNormal("sigma", sigma=5)
    obs = pm.Normal("obs", mu=mu, sigma=sigma, observed=observed_spreads)
    
    # 3. Sampling (MCMC)
    # This explores the probability space to find the best curve
    trace = pm.sample(1000, tune=1000, target_accept=0.9)

    import matplotlib.pyplot as plt

# 1. Setup Plot Style
plt.style.use('ggplot') 
fig, ax = plt.subplots(figsize=(12, 6))

# 2. Generate Prediction Range
x_plot = np.linspace(0.5, 12, 200)
X_plot = get_bspline_matrix(x_plot, knots)
post_beta = trace_val.posterior["beta"].stack(sample=("chain", "draw")).values
curve_samples = np.dot(X_plot, post_beta)

# 3. Plot the "Uncertainty Cloud" (The Wow Factor)
ax.fill_between(x_plot, 
                np.percentile(curve_samples, 2.5, axis=1), 
                np.percentile(curve_samples, 97.5, axis=1), 
                color='skyblue', alpha=0.3, label="95% Credible Interval")

# 4. Plot the Mean Line
ax.plot(x_plot, curve_samples.mean(axis=1), color='navy', lw=2, label="Bayesian Mean Curve")

# 5. Plot the Market Points
ax.scatter(tenors, spreads, color='crimson', s=100, edgecolors='white', zorder=5, label="Market Quotes")

# 6. Formatting for AdComs
ax.set_title("Malaysian Corporate Credit Curve (Bayesian P-Spline)", fontsize=15, pad=20)
ax.set_xlabel("Tenor (Years)")
ax.set_ylabel("Spread (bps)")
ax.legend(loc='upper left')

plt.tight_layout()
plt.show()